# Retail E-Commerce Customer-Support Chatbot — FLAN-T5 Fine-Tuning

**Author:** Harish Chandra · Master's in CS: AI & ML, Woolf University
*Industry Immersion Capstone — extends the Deep Learning for NLP chatbot project.*

This notebook fine-tunes `google/flan-t5-small` on the **Bitext retail e-commerce** customer-support dataset, evaluates it against the un-tuned baseline, and ships a Gradio demo.
**Architecture: fine-tuning only (no RAG).** Run on a **T4 GPU** (Runtime > Change runtime type > T4 GPU), then *Run all*.

> **After running, download these into the final `Main/` structure:**
> - `retail-ecommerce-chatbot-v2/` → `Main/chatbot/`
> - `bitext-retail-ecommerce-...csv` → `Main/chatbot/`
> - `figures/*.png` → `Main/research paper/resources/figures/`
> - `artifacts/*` (metrics.json, qualitative_eval.csv, comparison_metrics.csv) → `Main/research paper/resources/`

## 1. Setup

In [ ]:
# Pinned for reproducibility (same stack as the original project, minus the RAG libraries)
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.33.0
!pip install -q evaluate==0.4.2 rouge_score sacrebleu bert_score
!pip install -q gradio==4.44.0
!pip install -q matplotlib seaborn scikit-learn pandas

In [ ]:
import os, time, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

warnings.filterwarnings("ignore")
random.seed(42); np.random.seed(42); torch.manual_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Mount Drive & configure output paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Everything is written here, then downloaded into the local Main/ structure.
BASE_DIR = "/content/drive/MyDrive/Capstone Retail Ecommerce"
SAVE_DIR = f"{BASE_DIR}/retail-ecommerce-chatbot-v2"   # -> Main/chatbot/
FIG_DIR  = f"{BASE_DIR}/figures"                        # -> Main/research paper/resources/figures/
ART_DIR  = f"{BASE_DIR}/artifacts"                      # -> Main/research paper/resources/
CSV_PATH = f"{BASE_DIR}/bitext-retail-ecommerce-llm-chatbot-training-dataset.csv"  # -> Main/chatbot/
for d in [BASE_DIR, SAVE_DIR, FIG_DIR, ART_DIR]:
    os.makedirs(d, exist_ok=True)
print("Output base:", BASE_DIR)

## 3. Data Loading

In [ ]:
from datasets import load_dataset

ds = load_dataset("bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset", split="train")
df_raw = ds.to_pandas()
df_raw.to_csv(CSV_PATH, index=False)   # saved copy for Main/chatbot/

print(f"Raw rows: {len(df_raw):,}")
print(f"Columns:  {list(df_raw.columns)}")
df_raw.head(3)

In [ ]:
print("Schema overview:")
df_raw.info()
print("\nUnique categories:", df_raw['category'].nunique())
print("Unique intents:   ", df_raw['intent'].nunique())

## 4. Exploratory Data Analysis

In [ ]:
# 4.1 Category distribution - breadth of e-commerce support coverage
plt.figure(figsize=(10, 5))
cat_counts = df_raw['category'].value_counts()
sns.barplot(x=cat_counts.index, y=cat_counts.values, color="#2a6f97")
plt.title("Retail E-Commerce Category Distribution")
plt.ylabel("# examples"); plt.xlabel("Category")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig1_category_distribution.png", dpi=300, bbox_inches='tight')
plt.show()
print("Categories:", list(cat_counts.index))

In [ ]:
# 4.2 Intent diversity - top 20 intents
plt.figure(figsize=(10, 6))
intent_counts = df_raw['intent'].value_counts().head(20)
sns.barplot(y=intent_counts.index, x=intent_counts.values, color="#0a9396")
plt.title("Top 20 Retail E-Commerce Intents")
plt.xlabel("# examples"); plt.ylabel("")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig2_intent_distribution.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Total unique intents: {df_raw['intent'].nunique()}")

In [ ]:
# 4.3 Token-length distributions - justify max_input=128, max_target=256
from transformers import AutoTokenizer
_tk = AutoTokenizer.from_pretrained("google/flan-t5-small")

sample = df_raw.dropna(subset=['instruction','response']).sample(min(5000, len(df_raw)), random_state=42)
inst_lens = sample['instruction'].astype(str).apply(lambda s: len(_tk.encode(s)))
resp_lens = sample['response'].astype(str).apply(lambda s: len(_tk.encode(s)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(inst_lens, bins=40, color="#ee9b00", edgecolor='white')
axes[0].axvline(128, color='red', linestyle='--', label='128')
axes[0].set_title("Instruction token lengths"); axes[0].set_xlabel("# tokens"); axes[0].legend()
axes[1].hist(resp_lens, bins=40, color="#bb3e03", edgecolor='white')
axes[1].axvline(256, color='red', linestyle='--', label='256')
axes[1].set_title("Response token lengths"); axes[1].set_xlabel("# tokens"); axes[1].legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig3_token_lengths.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"Instruction tokens - mean {inst_lens.mean():.1f}, p95 {np.percentile(inst_lens,95):.0f}, max {inst_lens.max()}")
print(f"Response tokens    - mean {resp_lens.mean():.1f}, p95 {np.percentile(resp_lens,95):.0f}, max {resp_lens.max()}")
print(f"Coverage @128 instructions: {(inst_lens<=128).mean():.1%} | @256 responses: {(resp_lens<=256).mean():.1%}")

In [ ]:
# 4.4 Sample query/response pairs from the top categories
for cat in cat_counts.index[:3]:
    print(f"\n{'='*70}\nCategory: {cat}\n{'='*70}")
    rows = df_raw[df_raw['category']==cat].dropna(subset=['instruction','response']).sample(2, random_state=42)
    for _, r in rows.iterrows():
        ans = str(r['response'])
        print(f"\nQ: {r['instruction']}")
        print(f"A: {ans[:300]}{'...' if len(ans)>300 else ''}")

## 5. Preprocessing & Cleaning

In [ ]:
before = len(df_raw)
df = df_raw[['instruction','response','category','intent']].copy().dropna()
n_dropna = len(df)
df = df[df['instruction'].astype(str).str.strip() != ""]
df = df[df['response'].astype(str).str.strip() != ""]
n_empty = len(df)
df = df.drop_duplicates(subset=['instruction','response'])
n_dedup = len(df)
df = df.rename(columns={"instruction":"input","response":"output"}).reset_index(drop=True)

print(f"{'Step':<32}{'Rows':>10}{'Removed':>12}")
print("-"*54)
print(f"{'Raw':<32}{before:>10,}{'-':>12}")
print(f"{'After dropna':<32}{n_dropna:>10,}{before-n_dropna:>12,}")
print(f"{'After empty filter':<32}{n_empty:>10,}{n_dropna-n_empty:>12,}")
print(f"{'After dedup':<32}{n_dedup:>10,}{n_empty-n_dedup:>12,}")
print(f"{'Final usable':<32}{len(df):>10,}{before-len(df):>12,}")
df.head(3)

In [ ]:
from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['category'])
val_df, test_df  = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['category'])
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

## 6. Tokenization

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import Dataset

MODEL_NAME = "google/flan-t5-small"
MAX_INPUT, MAX_TARGET = 128, 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print(f"Loaded {MODEL_NAME} - {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

def tokenize(batch):
    enc = tokenizer(batch["input"], max_length=MAX_INPUT, truncation=True, padding="max_length")
    lab = tokenizer(text_target=batch["output"], max_length=MAX_TARGET, truncation=True, padding="max_length")
    # mask pad tokens in labels so they do not contribute to the loss
    enc["labels"] = [[(t if t != tokenizer.pad_token_id else -100) for t in seq] for seq in lab["input_ids"]]
    return enc

drop_cols = ["input","output","category","intent"]
train_ds = Dataset.from_pandas(train_df, preserve_index=False).map(tokenize, batched=True, remove_columns=drop_cols)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False).map(tokenize, batched=True, remove_columns=drop_cols)
print("Tokenized -> train:", len(train_ds), "| val:", len(val_ds))

## 7. Fine-Tuning

In [ ]:
from transformers import (Seq2SeqTrainingArguments, Seq2SeqTrainer,
                           DataCollatorForSeq2Seq, EarlyStoppingCallback)

# NOTE: FLAN-T5 overflows in FP16 -> NaN loss; the T4 has no bf16, so we train in FP32.
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/ft_ckpt",
    overwrite_output_dir=True,
    num_train_epochs=5,                 # guideline cap is 25; early stopping usually halts sooner
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,      # effective batch size = 16
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_steps=500,
    lr_scheduler_type="linear",
    predict_with_generate=True,
    eval_strategy="steps", eval_steps=500,
    save_strategy="steps", save_steps=500, save_total_limit=2,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False,
    fp16=False, bf16=False,
    report_to="none", seed=42,
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model)
trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
train_result = trainer.train()
print(train_result)

In [ ]:
# 7.1 Learning curves
log = trainer.state.log_history
ts, tl, es, el = [], [], [], []
for e in log:
    if 'loss' in e and 'eval_loss' not in e: ts.append(e['step']); tl.append(e['loss'])
    if 'eval_loss' in e: es.append(e['step']); el.append(e['eval_loss'])

plt.figure(figsize=(10,5))
plt.plot(ts, tl, label="Training loss", alpha=0.8)
plt.plot(es, el, label="Validation loss", marker='o', color='crimson')
plt.xlabel("Step"); plt.ylabel("Loss")
plt.title("Learning Curves - FLAN-T5 fine-tuning (retail e-commerce)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig4_training_loss.png", dpi=300, bbox_inches='tight')
plt.show()
if el:
    print(f"Final train {tl[-1]:.4f} | final val {el[-1]:.4f} | best val {min(el):.4f} @ step {es[int(np.argmin(el))]}")

In [ ]:
# 7.2 Save fine-tuned model -> download to Main/chatbot/retail-ecommerce-chatbot-v2/
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved model to:", SAVE_DIR)

## 8. Evaluation (baseline vs fine-tuned)

In [ ]:
import evaluate as hf_eval
rouge_metric = hf_eval.load("rouge")
bleu_metric  = hf_eval.load("sacrebleu")
USE_BERTSCORE = True   # set False to skip (downloads a ~400MB model)
bertscore_metric = hf_eval.load("bertscore") if USE_BERTSCORE else None

In [ ]:
EVAL_N = 300
eval_subset = test_df.sample(min(EVAL_N, len(test_df)), random_state=42).reset_index(drop=True)
queries    = eval_subset['input'].tolist()
references = eval_subset['output'].tolist()
print(f"Evaluating on {len(eval_subset)} held-out test samples")

In [ ]:
def batch_generate(mdl, tok, prompts, max_new_tokens=256, batch_size=16):
    mdl.eval(); mdl.to(DEVICE)
    out_all = []
    for i in range(0, len(prompts), batch_size):
        enc = tok(prompts[i:i+batch_size], padding=True, truncation=True,
                  max_length=128, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            gen = mdl.generate(**enc, max_new_tokens=max_new_tokens, num_beams=4,
                               no_repeat_ngram_size=3, early_stopping=True)
        out_all.extend(tok.batch_decode(gen, skip_special_tokens=True))
    return out_all

def compute_metrics(preds, refs, label):
    rouge = rouge_metric.compute(predictions=preds, references=refs, use_stemmer=True)
    bleu  = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])
    res = {"model": label,
           "ROUGE-1": round(rouge['rouge1'],4), "ROUGE-2": round(rouge['rouge2'],4),
           "ROUGE-L": round(rouge['rougeL'],4), "BLEU": round(bleu['score'],4),
           "avg_response_words": round(float(np.mean([len(p.split()) for p in preds])),2)}
    if USE_BERTSCORE:
        bs = bertscore_metric.compute(predictions=preds, references=refs, lang="en", verbose=False)
        res["BERTScore-F1"] = round(float(np.mean(bs['f1'])),4)
    return res

In [ ]:
# 8.1 Zero-shot baseline (un-tuned flan-t5-small)
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
base_tok   = AutoTokenizer.from_pretrained(MODEL_NAME)
t0 = time.time(); base_preds = batch_generate(base_model, base_tok, queries); bt = time.time()-t0
baseline_metrics = compute_metrics(base_preds, references, "zero-shot baseline")
baseline_metrics["latency_ms_per_query"] = round(1000*bt/len(queries),1)
print(baseline_metrics)

In [ ]:
# 8.2 Fine-tuned model
t0 = time.time(); ft_preds = batch_generate(model, tokenizer, queries); ft = time.time()-t0
ft_metrics = compute_metrics(ft_preds, references, "fine-tuned")
ft_metrics["latency_ms_per_query"] = round(1000*ft/len(queries),1)
print(ft_metrics)

In [ ]:
# 8.3 Comparison table + save metrics
comp_df = pd.DataFrame([baseline_metrics, ft_metrics])
comp_df.to_csv(f"{ART_DIR}/comparison_metrics.csv", index=False)
with open(f"{ART_DIR}/metrics.json","w") as f:
    json.dump({"baseline": baseline_metrics, "fine_tuned": ft_metrics}, f, indent=2)
print("Saved metrics.json + comparison_metrics.csv")
comp_df

In [ ]:
# 8.4 Metric comparison bar chart
mset = ["ROUGE-1","ROUGE-2","ROUGE-L"]
x = np.arange(len(mset)); w = 0.35
plt.figure(figsize=(9,5))
plt.bar(x-w/2, [baseline_metrics[m] for m in mset], w, label="baseline", color="#94a3b8")
plt.bar(x+w/2, [ft_metrics[m]       for m in mset], w, label="fine-tuned", color="#0a9396")
plt.xticks(x, mset); plt.ylabel("score"); plt.title("Baseline vs Fine-tuned (ROUGE)")
plt.legend(); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig5_metric_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 8.5 Per-category ROUGE-L (fine-tuned) - where the bot is strong / weak
percat = {}
for cat in sorted(eval_subset['category'].unique()):
    idx = eval_subset.index[eval_subset['category']==cat].tolist()
    if len(idx) < 3:
        continue
    r = rouge_metric.compute(predictions=[ft_preds[i] for i in idx],
                             references=[references[i] for i in idx], use_stemmer=True)
    percat[cat] = round(r['rougeL'],4)
percat = dict(sorted(percat.items(), key=lambda kv: kv[1], reverse=True))

plt.figure(figsize=(10,5))
sns.barplot(x=list(percat.values()), y=list(percat.keys()), color="#ee9b00")
plt.title("Per-Category ROUGE-L (fine-tuned)"); plt.xlabel("ROUGE-L")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig6_per_category_rougeL.png", dpi=300, bbox_inches='tight')
plt.show()
print(json.dumps(percat, indent=2))

## 9. Qualitative Test Battery

In [ ]:
test_prompts = [
    ("in-domain", "How can I track my order?"),
    ("in-domain", "I want to cancel my order."),
    ("in-domain", "How do I return a product?"),
    ("in-domain", "How do I get a refund?"),
    ("in-domain", "What are the delivery options?"),
    ("in-domain", "How long does shipping take?"),
    ("in-domain", "How do I change my shipping address?"),
    ("in-domain", "I want to place an order."),
    ("in-domain", "How can I see my invoice?"),
    ("in-domain", "I want to file a complaint about a product."),
    ("out-of-domain", "What is the weather like today?"),
    ("out-of-domain", "Tell me a joke."),
    ("out-of-domain", "Who won the football match yesterday?"),
    ("ambiguous", "help me"),
    ("ambiguous", "order"),
    ("ambiguous", "refund"),
]
rows = []
for cat, p in test_prompts:
    rows.append({"category": cat, "prompt": p,
                 "baseline (zero-shot)": batch_generate(base_model, base_tok, [p])[0],
                 "fine-tuned": batch_generate(model, tokenizer, [p])[0]})
qual_df = pd.DataFrame(rows)
qual_df.to_csv(f"{ART_DIR}/qualitative_eval.csv", index=False)
print("Saved qualitative_eval.csv")
qual_df

### 9.1 Failure-case analysis (fill in after running)
Inspect the table above and note 2-3 weak cases (e.g., out-of-domain prompts answered confidently but wrongly, or single-word ambiguous prompts). Use these in the paper's *Limitations* (Phase 2) and the video's Part 7.

## 10. Interactive Demo (Gradio)

In [ ]:
import gradio as gr

def respond(message, history):
    if not message or not message.strip():
        return "Please enter a question."
    return batch_generate(model, tokenizer, [message.strip()])[0]

demo = gr.ChatInterface(
    fn=respond,
    title="Retail E-Commerce Support Bot (FLAN-T5, fine-tuned)",
    description="Ask about orders, refunds, shipping, returns, payments, or your account.",
    examples=["How can I track my order?", "How do I get a refund?",
              "I want to cancel my order.", "What are the delivery options?"],
)
demo.launch(share=True, debug=False)

## 11. Results & Discussion (notes to transfer into the paper)
Fill these in from the cells above, then write them up in your own words (Phase 2 / Phase 3 originality rules):
- Baseline vs fine-tuned: ROUGE-L `[__]` -> `[__]`, BLEU `[__]` -> `[__]` (improvement `[__]`).
- Strongest categories: `[__]`; weakest: `[__]` (from fig6).
- Avg inference latency: `[__]` ms/query on T4.
- Overfitting behavior from the loss curve (fig4): `[__]`.

## 12. Limitations & Future Work (notes)
- flan-t5-small is small -> can be generic on rare/nuanced queries.
- Dataset is synthetic -> may not capture real customer phrasing.
- ROUGE/BLEU approximate quality; no human evaluation yet.
- Future: larger model, human eval, real anonymized logs, multilingual, production guardrails.

## 13. References
- Bitext, *Retail E-Commerce LLM Chatbot Training Dataset*, Hugging Face (CDLA-Sharing-1.0).
- C. Raffel et al., "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer (T5)," JMLR, 2020.
- H. W. Chung et al., "Scaling Instruction-Finetuned Language Models (FLAN)," 2022.
- A. Vaswani et al., "Attention Is All You Need," NeurIPS, 2017.
- (Add the domain/e-commerce-chatbot references gathered in Phase 2.)